## Sensitivity Computation for the Design of Experiments

Recall that `Pyomo.DoE`  utilizes the sensitivity matrix defined as the derivative of the estimates of the measured outputs $\hat{y}$ with respect to the parameter to be estimated $\hat{\theta}$:
$$
Q = \nabla_{\hat{\theta}} \hat{y}
$$
to compute the FIM as:
$$
M = Q^T \Sigma^{-1}Q 
$$
where $\Sigma$ is the measurement covariance matrix

Pyomo.DoE offers two different approaches to computing the sensitivity matrix:
1. Finite difference method: With Central finite difference method that utilizes parameter perturbations for estimating the sensitivity matrix as: $$Q \approx \frac{\hat{y}(\hat{\theta} + \Delta \hat{\theta})- \hat{y}(\hat{\theta} - \Delta \hat{\theta})}{2 \Delta \hat{\theta}}$$ where $\Delta \hat{\theta}$ is the step size
2. Automatic differentiation method: Pyomo.DoE uses automatic differentiation in PyNumero to compute sensitivities directly from the model equations, avoiding repeated parameter perturbation solves: $$\nabla_{\hat{\theta}}F = \begin{bmatrix}\nabla_{\hat{y}}F\\\nabla_{\hat{x}}F\end{bmatrix}\begin{bmatrix}\nabla_{\hat{\theta}}\hat{y}\\\nabla_{\hat{\theta}}\hat{x}\end{bmatrix}$$ where $F(\hat{x},\hat{y}, \hat{\theta}, u) = 0$ is the model of the system



Now that we have introduced the two sensitivity calculation methods, we compare their computational cost. Both approaches produce the sensitivity matrix needed to assemble the FIM, but they differ in how many model evaluations they require.

The finite-difference approach estimates sensitivities by repeatedly perturbing parameters and resolving the model. This can become expensive as the number of parameters grows. In contrast, the automatic differentiation approach computes sensitivities directly from the model equations using PyNumero, avoiding repeated finite-difference solves.

Next, we time both methods on the TC Lab example and compare the resulting FIM calculations.

In [ ]:
import sys

# If running on Google Colab, install Pyomo and Ipopt via IDAES
on_colab = "google.colab" in sys.modules
if on_colab:
    !wget "https://raw.githubusercontent.com/dowlinglab/pyomo-doe/main/notebooks/tclab_pyomo.py"

# import TCLab model, simulation, and data analysis functions
from tclab_pyomo import (
    TC_Lab_data,
    TC_Lab_experiment,
    extract_results,
    extract_plot_results,
    results_summary,
)

# set default number of states in the TCLab model
number_tclab_states = 2

## Load experimental data (sine test)

We will load the sine test data to serve as an initial point. Recall our create model function will use supplied data to initialize the Pyomo model. Carefully initialization is often required for optimization of large-scale dynamic systems.

In [ ]:
import pandas as pd

if on_colab:
    file = "https://raw.githubusercontent.com/dowlinglab/pyomo-doe/main/data/tclab_sine_test_5min_period.csv"
else:
    file = '../data/tclab_sine_test_5min_period.csv'
df = pd.read_csv(file)
df.head()

We will create the data object for the design of experiment problems that follow

In [ ]:
# Here, we will induce a step size of 15 seconds, as to not give too many 
# degrees of freedom for experimental design.
skip =15

# Create the data object considering the new control points every 6 seconds
tc_data = TC_Lab_data(
    name="Sine Wave Test for Heater 1",
    time=df['Time'].values[::skip],
    T1=df['T1'].values[::skip],
    u1=df['Q1'].values[::skip],
    P1=200,
    TS1_data=None,
    T2=df['T2'].values[::skip],
    u2=df['Q2'].values[::skip],
    P2=200,
    TS2_data=None,
    Tamb=df['T1'].values[0],
)

## Calculate FIM at initial point (Values from $L_2$ regularization)

We will start by re-computing the prior FIM from $L_2$ regularization

In [ ]:
# Load Pyomo.DoE class
from pyomo.contrib.doe import DesignOfExperiments

from pyomo.environ import SolverFactory

# Copied from previous notebook
theta_values = {
    'Ua': 0.041705,
    'Ub': 0.012009,
    'inv_CpH': 0.167457,
    'inv_CpS': 4.545432,
}

In [ ]:
# Create experiment object for design of experiments
doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values, number_of_states=number_tclab_states)

# Create the design of experiments object using our experiment instance from above
TC_Lab_DoE = DesignOfExperiments(experiment=doe_experiment, 
                                 step=1e-2,
                                 scale_constant_value=1,
                                 scale_nominal_param_value=True, 
                                 tee=True,)

We will define a prior FIM from $L_2$ regularization 

In [ ]:

import numpy as np

cov = np.array([
    [1.857017e-10, -2.576198e-10, 1.402148e-09, -2.242347e-12],
    [-2.576198e-10, 1.624383e-07, 9.10987e-08, -6.32555e-05],
    [1.402148e-09, 9.10987e-08, 1.031454e-07, -3.890789e-05],
    [-2.242347e-12, -6.325555e-05, -3.890789e-05, 2.499914e-02],
])

FIM = np.linalg.inv(cov)

# Make sure resultant FIM is symmetric within numerical tolerance

FIM = 0.5*(FIM + FIM.T)





## Optimize next experiment (D-optimality) using central finite difference method

We are now ready to solve the next experiment design problem. Here, we create a new `DesignOfExperiments` object and pass in the `prior_FIM`, which represents the information already collected from the previous experiment. By optimizing with this prior information included, Pyomo.DoE identifies the next best experiment to run according to the selected design criterion. In this section, we use D-optimality by setting `objective_option="determinant"`.

In [ ]:
# Create experiment object for design of experiments
doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values, number_of_states=number_tclab_states)
solver = SolverFactory("ipopt")
solver.options["max_iter"] = 3000
solver.options["tol"] = 1e-6
solver.options["linear_solver"] = "ma57"
solver.options["nlp_scaling_method"] = "gradient-based"
solver.options["acceptable_tol"] = 1e-4
# Create the design of experiments object using our experiment instance from above
TC_Lab_DoE_D = DesignOfExperiments(experiment=doe_experiment, 
                                 step=1e-2,
                                 scale_constant_value=1,
                                 scale_nominal_param_value=True,
                                 objective_option="determinant",  # Now we specify a type of objective, D-opt = "determinant"
                                 prior_FIM=FIM,  # We use the prior information from the existing experiment!,,,,,,,,,,,,,,,,,,,,,,,
                                 tee=True,
                                 solver = solver,)


TC_Lab_DoE_D.run_doe()

In [ ]:
import numpy as np, hashlib, json

def h(a):
    a = np.asarray(a, dtype=float)
    return hashlib.sha256(a.tobytes()).hexdigest()[:16]

print("theta_values:", theta_values)
print("theta hash:", h(list(theta_values.values()) if isinstance(theta_values, dict) else theta_values))
print("time len:", len(tc_data.time), "time[0],time[-1]:", float(tc_data.time[0]), float(tc_data.time[-1]))
print("u1 hash:", h(tc_data.u1), "T1 hash:", h(tc_data.T1), "T2 hash:", h(tc_data.T2))
print("prior_FIM hash:", h(FIM))
print("prior eig:", np.linalg.eigvalsh(0.5*(np.asarray(FIM)+np.asarray(FIM).T)))
print("solver options:", json.dumps(dict(solver.options), sort_keys=True))
print("DoE gradient method:", getattr(TC_Lab_DoE_D, "_gradient_method", None))
print("DoE fd_formula:", getattr(TC_Lab_DoE_D, "fd_formula", None))
print("DoE Cholesky_option:", getattr(TC_Lab_DoE_D, "Cholesky_option", None))
print("DoE use_grey_box:", getattr(TC_Lab_DoE_D, "use_grey_box", None))


## Run D-optimality experiment using automatic differentiation

In [ ]:

# Create experiment object for design of experiments
# Create experiment object for design of experiments
doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values, number_of_states=number_tclab_states)

# Create the design of experiments object using our experiment instance from above
TC_Lab_DoE_D_AD = DesignOfExperiments(experiment=doe_experiment, 
                                 step=1e-2,
                                 scale_constant_value=1,
                                 scale_nominal_param_value=True,
                                 gradient_method = "pynumero",
                                 objective_option="determinant",  # Now we specify a type of objective, D-opt = "determinant"
                                 prior_FIM=FIM,  # We use the prior information from the existing experiment!,,,,,,,,,,,,,,,,,,,,,,,
                                 tee=False,
                                 solver = solver,)


TC_Lab_DoE_D_AD.run_doe()

Let us check whether the two sensitivity approaches give consistent optimization results. If automatic differentiation is going to replace central finite differences, it should produce a comparable D-optimal design, not just run faster.

We compare the resulting FIMs using two quantities. First, we compare the D-optimal objective value, which measures the determinant-based information content of the FIM. Second, we compare the condition number,

$$
\kappa(\mathbf{M}) =
\frac{\lambda_{\max}(\mathbf{M})}{\lambda_{\min}(\mathbf{M})},
$$

which measures how evenly information is distributed across parameter directions. Similar objective values and similar condition numbers indicate that the two methods produce comparable FIM quality.

In [ ]:
import numpy as np
import pandas as pd


def fim_accuracy_metrics(FIM):
    """Compute objective and conditioning metrics for a Fisher information matrix.

    Parameters
    ----------
    FIM : array-like
        Fisher information matrix from a Pyomo.DoE result object.

    Returns
    -------
    dict
        Dictionary containing the D-optimal objective value, minimum eigenvalue,
        maximum eigenvalue, and condition number.
    """
    FIM_array = np.array(FIM, dtype=float)
    eigvals = np.linalg.eigvalsh(FIM_array)

    lambda_min = np.min(eigvals)
    lambda_max = np.max(eigvals)
    condition_number = lambda_max / lambda_min

    return {
        "log10 det(FIM)": np.log10(np.linalg.det(FIM_array)),
        "lambda_min": lambda_min,
        "lambda_max": lambda_max,
        "condition_number": condition_number,
    }


accuracy_df = pd.DataFrame(
    {
        "Central finite difference": fim_accuracy_metrics(TC_Lab_DoE_D.results["FIM"]),
        "Automatic differentiation": fim_accuracy_metrics(TC_Lab_DoE_D_AD.results["FIM"]),
    }
).T

accuracy_df = accuracy_df.round(
    {
        "log10 det(FIM)": 6,
        "lambda_min": 2,
        "lambda_max": 2,
        "condition_number": 2,
    }
)

print("=== Accuracy comparison: central finite difference vs automatic differentiation ===")
display(accuracy_df)

The central finite difference and automatic differentiation approaches give very similar FIM quality. The D-optimal objective values are nearly the same: $30.318$ for central finite differences and $30.315$ for automatic differentiation. The conditioning metrics are also comparable, with condition numbers of about $5.26 \times 10^5$ and $5.29 \times 10^5$, respectively.

This indicates that automatic differentiation produces essentially the same optimized experiment quality as central finite differences for this example. Having checked that the two methods give comparable accuracy, we can now compare their computational cost.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

central_time_metrics = {
    "Build Time": TC_Lab_DoE_D.results.get("Build Time"),
    "Initialization Time": TC_Lab_DoE_D.results.get("Initialization Time"),
    "Solve Time": TC_Lab_DoE_D.results.get("Solve Time"),
    "Wall-clock Time": TC_Lab_DoE_D.results.get("Wall-clock Time"),
}

AD_time_metrics = {
    "Build Time": TC_Lab_DoE_D_AD.results.get("Build Time"),
    "Initialization Time": TC_Lab_DoE_D_AD.results.get("Initialization Time"),
    "Solve Time": TC_Lab_DoE_D_AD.results.get("Solve Time"),
    "Wall-clock Time": TC_Lab_DoE_D_AD.results.get("Wall-clock Time"),
}

labels = []
central_vals = []
AD_vals = []

for key in central_time_metrics:
    c = central_time_metrics.get(key)
    s = AD_time_metrics.get(key)
    if c is None or s is None:
        continue
    try:
        c = float(c)
        s = float(s)
    except (TypeError, ValueError):
        continue
    labels.append(key)
    central_vals.append(c)
    AD_vals.append(s)

x = np.arange(len(labels))
width = 0.36

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, central_vals, width, label="Central finite difference", color="#b7cde8")
plt.bar(x + width/2, AD_vals, width, label="Automatic Differentiation", color="#4f86c6")

plt.xticks(x, labels, rotation=30, ha="right")
plt.ylabel("Time (s)")
plt.title("Time comparison: Central finite difference vs Automatic differentiation")
plt.legend()
plt.tight_layout()
plt.show()


Automatic differentiation gives us the same type of sensitivity information needed for the FIM, but with less computational overhead than central finite differences.